In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
from tensorflow.keras.utils import plot_model

# --- Constants ---
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 50   # train longer since no pretraining

# --- Data Preprocessing ---
def preprocess_data(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    # Data augmentation
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    return image, label

# --- Residual Block ---
def residual_block(x, filters, stride=1):
    shortcut = x
    x = layers.Conv2D(filters, (3,3), strides=stride, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters, (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x)

    # match dimensions if stride > 1
    if stride > 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1,1), strides=stride, padding="same")(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([shortcut, x])
    x = layers.Activation("relu")(x)
    return x

# --- Custom Model ---
def create_custom_model(num_classes):
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    # Stem
    x = layers.Conv2D(32, (7,7), strides=2, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((3,3), strides=2, padding="same")(x)

    # Residual Stages
    x = residual_block(x, 64)
    x = residual_block(x, 64)

    x = residual_block(x, 128, stride=2)
    x = residual_block(x, 128)

    x = residual_block(x, 256, stride=2)
    x = residual_block(x, 256)

    x = residual_block(x, 512, stride=2)
    x = residual_block(x, 512)

    # Classification Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return models.Model(inputs, outputs)

# --- Load Dataset ---
train_ds = tf.keras.utils.image_dataset_from_directory(
    "/Users/uditbhardwaj/Documents/AndroidProjects/Model/food-101/images",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="training",
    seed=123
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    "/Users/uditbhardwaj/Documents/AndroidProjects/Model/food-101/images",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="validation",
    seed=123
)

class_names = train_ds.class_names
num_classes = len(class_names)

# Apply preprocessin
train_ds = train_ds.map(preprocess_data).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess_data).prefetch(tf.data.AUTOTUNE)

# --- Build & Compile ---
model = create_custom_model(num_classes)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# --- Show Model ---
model.summary()
plot_model(model, to_file="custom_model.png", show_shapes=True, show_layer_names=True)

# --- Train ---
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)
    ]
)

# --- Plot Training History ---
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.title("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title("Loss")

plt.savefig("custom_training_curves.png")
plt.show()

Found 101000 files belonging to 101 classes.
Using 80800 files for training.
Found 101000 files belonging to 101 classes.
Using 20200 files for validation.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 64, 64,    │      4,736 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 64, 64,    │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 32, 32,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 32, 32,    │      2,112 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │     36,928 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 32, 32,    │          0 │ add[0][0]         │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 32, 32,    │     36,928 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 32, 32,    │     36,928 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_5[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 32, 32,    │          0 │ activation[0][0]

 Total params: 11,221,797 (42.81 MB)

 Trainable params: 11,212,133 (42.77 MB)

 Non-trainable params: 9,664 (37.75 KB)

Epoch 1/50
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 1740s 686ms/step - accuracy: 0.0457 - loss: 4.5392 - val_accuracy: 0.0699 - val_loss: 4.6188 - learning_rate: 0.0010
Epoch 2/50
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 2428s 962ms/step - accuracy: 0.1488 - loss: 3.6281 - val_accuracy: 0.1637 - val_loss: 3.7224 - learning_rate: 0.0010
Epoch 3/50
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 3766s 1s/step - accuracy: 0.2339 - loss: 3.1772 - val_accuracy: 0.2363 - val_loss: 3.2197 - learning_rate: 0.0010
Epoch 4/50
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 3786s 1s/step - accuracy: 0.2952 - loss: 2.8704 - val_accuracy: 0.3100 - val_loss: 2.8386 - learning_rate: 0.0010
Epoch 5/50
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 3818s 2s/step - accuracy: 0.3498 - loss: 2.6194 - val_accuracy: 0.3117 - val_loss: 2.8560 - learning_rate: 0.0010
Epoch 6/50
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 2833s 1s/step - accuracy: 0.3955 - loss: 2.4011 - val_accuracy: 0.3522 - val_loss: 2.6882 - learning_rate: 0.0010
Epoch 7/50
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 4869s 2s/step - 

KeyboardInterrupt: 

In [3]:
# Save the trained model first
model.save("best_model.h5")

from tensorflow.keras.models import load_model
best_model = load_model("best_model.h5")